# Pipeline End-to-End de Validacao de Dados para ML

Este notebook integra os quatro pilares de validacao de dados ensinados
na disciplina em uma pipeline completa:

| Estagio | Conceito | Aula |
|---------|----------|------|
| 1 | Expectativas declarativas | Aula 01 — Great Expectations |
| 2 | Schema tipado de entrada | Aula 02 — Pandera |
| 3 | Schema de predicoes | Aula 02 — Pandera |
| 4 | Validacao runtime de payloads | Aula 03 — Pydantic |
| 5 | Quality gate pass/warn/fail | Aula 04 — Pipeline Gates |

## Fluxo da pipeline

```
Ingestao → Expectation Suite → Schema de Entrada → Modelo
    → Schema de Predicoes → Serving → Validacao Runtime → Gate Final
```

In [ ]:
import logging
import json
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

from e2e_validation_pipeline import (
    generate_churn_dataset,
    simulate_predictions,
    build_serving_payloads,
    run_expectation_suite,
    validate_input_schema,
    validate_prediction_schema,
    validate_serving_payloads,
    run_quality_gate,
    run_pipeline,
    format_report,
)

## Cenario 1: Dados limpos

Executamos a pipeline completa com dados sinteticos **sem erros**.
Todos os estagios devem passar.

In [ ]:
report_clean = run_pipeline(inject_errors=False)
print(format_report(report_clean))

## Cenario 2: Dados com erros injetados

Agora injetamos problemas realistas no dataset:
- Valores negativos em tenure e monthly_charges
- Categorias invalidas em contract e churn
- Duplicatas
- Valores ausentes

A pipeline deve **detectar e reportar** cada problema.

In [ ]:
report_dirty = run_pipeline(inject_errors=True, random_state=99)
print(format_report(report_dirty))

## Comparacao entre cenarios

In [ ]:
import pandas as pd

comparison = pd.DataFrame([
    {"cenario": "Dados limpos",
     "decisao": report_clean.overall,
     "aprovados": report_clean.passed,
     "reprovados": report_clean.failed},
    {"cenario": "Dados com erros",
     "decisao": report_dirty.overall,
     "aprovados": report_dirty.passed,
     "reprovados": report_dirty.failed},
])
comparison

## Explorando cada estagio individualmente

Vamos executar cada estagio separadamente para entender os detalhes.

In [ ]:
# Gerar dados com erros
df_dirty = generate_churn_dataset(rows=200, inject_errors=True, random_state=99)
print(f"Shape: {df_dirty.shape}")
df_dirty.describe()

In [ ]:
# Estagio 1: Expectation Suite
s1 = run_expectation_suite("churn_dirty", df_dirty)
print(f"Resultado: {'PASS' if s1['passed'] else 'FAIL'}")
print(f"Expectativas: {s1['ok']}/{s1['total']}")
if s1['failures']:
    print(f"Falhas: {s1['failures']}")

In [ ]:
# Estagio 2: Input Schema
s2 = validate_input_schema(df_dirty)
print(f"Resultado: {'PASS' if s2['passed'] else 'FAIL'}")
print(f"Backend: {s2['backend']}")
if s2['errors']:
    for err in s2['errors']:
        print(f"  Erro: {err[:200]}...")

In [ ]:
# Estagio 5: Quality Gate
s5 = run_quality_gate(df_dirty)
print(f"Decisao: {s5['decision']}")
print(f"Checks:")
for name, check in s5['checks'].items():
    print(f"  {name}: {check['value']} ({check['status']})")

## Resumo

A pipeline end-to-end demonstra como combinar os quatro pilares de
validacao em um fluxo unico e coerente:

1. **Expectativas declarativas** detectam violacoes de contrato
2. **Schemas tipados** garantem estrutura e tipos corretos
3. **Validacao runtime** protege fronteiras de serving
4. **Quality gates** tomam a decisao final de pass/warn/fail

Em producao, essa pipeline seria integrada com:
- CI/CD (rodar antes de cada deploy)
- Monitoramento (alertas quando gates falham)
- Observabilidade (metricas de taxa de rejeicao)